# Submission 6 — Improved LightGBM + CatBoost Ensemble (Kaggle)

**Before running, in Kaggle notebook settings:**
- Accelerator: **GPU T4 x2** (CatBoost uses GPU; LightGBM runs on CPU)
- Internet: **Off** (competition rule)
- Add competition input: `home-credit-credit-risk-model-stability`

**What this notebook does:**
1. Loads the same features as Sub 2 / Sub 3 / Sub 5 (depth-0 + 4 depth-1 aggregations + 10% stability filter)
2. Loads improved LightGBM test predictions (from Sub 5 / notebook 06)
3. Trains a CatBoost model on the same data
4. Blends the two test-set probability outputs: `ensemble = 0.5 * lgbm + 0.5 * catboost`
5. Writes one combined `submission.csv`

**Why ensembling helps the Gini stability metric specifically:**
the two models make different errors → averaging reduces variance → lower residual std → higher score.

In [ ]:
import glob
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
from scipy.stats import linregress
import matplotlib.pyplot as plt
import gc
import os
import warnings
warnings.filterwarnings('ignore')

DATA = '/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files'
OUTPUT = '/kaggle/working'
TRAIN_CUTOFF = 60

print(os.listdir(f'{DATA}/train')[:10])

## 1. Aggregate depth-1 tables (lazy — low memory)

In [ ]:
def aggregate_lazy(pattern: str, name: str) -> pl.DataFrame:
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No files found: {pattern}')
    print(f'{name}: {len(files)} file(s)')

    ref_schema = None
    for f in files:
        s = pl.read_parquet(f, n_rows=10).schema
        if any(v != pl.Null for v in s.values()):
            ref_schema = s
            break

    parts = []
    for f in files:
        part = pl.read_parquet(f)
        exprs = []
        for c in part.columns:
            if c == 'case_id':
                continue
            dtype = part[c].dtype
            if dtype == pl.Null and ref_schema and c in ref_schema:
                target = ref_schema[c]
                exprs.append(pl.col(c).cast(target if target != pl.Null else pl.Utf8))
            elif dtype in [pl.Float64, pl.Int64, pl.Int32, pl.Int16, pl.Int8]:
                exprs.append(pl.col(c).cast(pl.Float32))
        if exprs:
            part = part.with_columns(exprs)
        parts.append(part)

    df = pl.concat(parts, how='diagonal_relaxed')
    del parts
    gc.collect()

    schema = df.schema
    num_cols = [c for c, dtype in schema.items() if c != 'case_id' and dtype == pl.Float32]
    cat_cols = [c for c, dtype in schema.items() if c != 'case_id' and dtype == pl.Utf8]

    aggs = [pl.len().alias(f'{name}_count').cast(pl.Int32)]
    for col in num_cols:
        aggs += [
            pl.col(col).mean().alias(f'{name}_{col}_mean'),
            pl.col(col).max().alias(f'{name}_{col}_max'),
            pl.col(col).min().alias(f'{name}_{col}_min'),
            pl.col(col).std().alias(f'{name}_{col}_std'),
        ]
    for col in cat_cols:
        aggs.append(pl.col(col).n_unique().cast(pl.Int16).alias(f'{name}_{col}_nunique'))

    result = df.group_by('case_id').agg(aggs)
    del df
    gc.collect()
    print(f'  -> {result.shape}')
    return result

In [ ]:
agg_applprev      = aggregate_lazy(f'{DATA}/train/train_applprev_1_*.parquet',        'applprev')
agg_cb_a          = aggregate_lazy(f'{DATA}/train/train_credit_bureau_a_1_*.parquet', 'cb_a')
agg_cb_b          = aggregate_lazy(f'{DATA}/train/train_credit_bureau_b_1.parquet',   'cb_b')
agg_person        = aggregate_lazy(f'{DATA}/train/train_person_1.parquet',            'person')

agg_applprev_test = aggregate_lazy(f'{DATA}/test/test_applprev_1_*.parquet',          'applprev')
agg_cb_a_test     = aggregate_lazy(f'{DATA}/test/test_credit_bureau_a_1_*.parquet',   'cb_a')
agg_cb_b_test     = aggregate_lazy(f'{DATA}/test/test_credit_bureau_b_1.parquet',     'cb_b')
agg_person_test   = aggregate_lazy(f'{DATA}/test/test_person_1.parquet',              'person')

## 2. Load depth-0 tables and join everything

In [ ]:
def cast_static(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col(c).cast(pl.Float32)
        for c in df.columns
        if df[c].dtype in [pl.Float64, pl.Int64, pl.Int32]
        and c not in ('case_id', 'target', 'WEEK_NUM', 'MONTH')
    ])

base      = pl.read_parquet(f'{DATA}/train/train_base.parquet')
static0   = pl.scan_parquet(sorted(glob.glob(f'{DATA}/train/train_static_0_*.parquet'))).collect().pipe(cast_static)
static_cb = pl.read_parquet(f'{DATA}/train/train_static_cb_0.parquet').pipe(cast_static)

test_base      = pl.read_parquet(f'{DATA}/test/test_base.parquet')
test_static0   = pl.scan_parquet(sorted(glob.glob(f'{DATA}/test/test_static_0_*.parquet'))).collect().pipe(cast_static)
test_static_cb = pl.read_parquet(f'{DATA}/test/test_static_cb_0.parquet').pipe(cast_static)

def build(base_df, s0, scb, aggs):
    df = base_df.join(s0, on='case_id', how='left').join(scb, on='case_id', how='left')
    for agg in aggs:
        df = df.join(agg, on='case_id', how='left')
    return df

df   = build(base, static0, static_cb, [agg_applprev, agg_cb_a, agg_cb_b, agg_person])
test = build(test_base, test_static0, test_static_cb, [agg_applprev_test, agg_cb_a_test, agg_cb_b_test, agg_person_test])

del static0, static_cb, test_static0, test_static_cb
del agg_applprev, agg_cb_a, agg_cb_b, agg_person
del agg_applprev_test, agg_cb_a_test, agg_cb_b_test, agg_person_test
gc.collect()

print(f'Train: {df.shape}')
print(f'Test:  {test.shape}')

## 3. Preprocessing

Same preprocessing for both models — string columns are integer-encoded (with `'NA'` for nulls).
LightGBM treats those integers as ordinal numeric; CatBoost gets them as `cat_features` and
treats them as unordered categories.

In [ ]:
DROP_COLS = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM', 'target']
TEST_DROP = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM']

def preprocess(df: pl.DataFrame, drop_cols: list):
    date_cols = [c for c in df.columns if c.endswith('_D') and c not in drop_cols]
    for col in date_cols:
        df = df.with_columns(
            pl.col(col).str.to_date(strict=False).cast(pl.Int32).alias(col)
        )

    # Treat both Utf8 strings AND Boolean columns as categoricals.
    # Boolean ones (_L suffix) become 'true'/'false'/'NA' strings, then int16 codes.
    # Without the Boolean branch they end up as pandas 'object' dtype → LightGBM rejects.
    cat_col_names = [
        c for c in df.columns
        if df[c].dtype in [pl.Utf8, pl.Boolean] and c not in drop_cols
    ]
    for col in cat_col_names:
        df = df.with_columns(
            pl.col(col).cast(pl.Utf8).fill_null('NA').cast(pl.Categorical).to_physical().cast(pl.Int16).alias(col)
        )

    df = df.drop([c for c in drop_cols if c in df.columns])
    return df, [c for c in cat_col_names if c in df.columns]

week_num = df['WEEK_NUM'].to_numpy()
y = df['target'].to_numpy()

X_df,      cat_cols      = preprocess(df,   DROP_COLS)
X_test_df, cat_cols_test = preprocess(test, TEST_DROP)
del df, test
gc.collect()

feature_cols = [c for c in X_df.columns if c in X_test_df.columns]
cat_cols     = [c for c in cat_cols if c in feature_cols]
X_df         = X_df.select(feature_cols)
X_test_df    = X_test_df.select(feature_cols)

print(f'Features before stability filter: {len(feature_cols)}')
print(f'Categorical columns (Utf8 + Boolean): {len(cat_cols)}')

## 4. Stability-first feature filtering

In [ ]:
train_mask = week_num <= TRAIN_CUTOFF

numeric_cols = [
    c for c in X_df.columns
    if X_df[c].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int16, pl.Int8]
    and c not in cat_cols
]

df_wk = (
    X_df.select(numeric_cols)
    .with_columns(pl.Series(name='WEEK_NUM', values=week_num.tolist()))
    .filter(pl.col('WEEK_NUM') <= TRAIN_CUTOFF)
)
weekly_means = df_wk.group_by('WEEK_NUM').agg([pl.col(c).mean() for c in numeric_cols]).sort('WEEK_NUM')

drift = {}
for col in numeric_cols:
    vals = weekly_means[col].drop_nulls().to_numpy()
    if len(vals) < 5 or np.abs(vals.mean()) < 1e-9:
        drift[col] = 0.0
    else:
        drift[col] = vals.std() / np.abs(vals.mean())

del df_wk, weekly_means
gc.collect()

drift_df = pl.DataFrame({'feature': list(drift.keys()), 'drift': list(drift.values())}).sort('drift', descending=True)
n_drop = max(1, int(len(drift_df) * 0.10))
unstable = set(drift_df.head(n_drop)['feature'].to_list())
stable_features = [c for c in feature_cols if c not in unstable]
cat_cols_stable = [c for c in cat_cols if c in stable_features]

print(f'Dropped {len(unstable)} unstable features, {len(stable_features)} remaining')
print(f'Categorical features remaining: {len(cat_cols_stable)}')

In [ ]:
X      = X_df.select(stable_features).to_pandas()
X_test = X_test_df.select(stable_features).to_pandas()
del X_df, X_test_df
gc.collect()

# Defensive cleanup: convert any object / bool / nullable dtypes that survived
# preprocessing into int16. Catches edge cases where a column was all-null in
# test (Polars Null dtype) or had an unusual dtype that the preprocess missed.
def cleanup_object_dtypes(df, cat_list):
    for c in df.columns:
        dt_str = str(df[c].dtype)
        if dt_str in ('object', 'bool', 'boolean', 'category') or 'Bool' in dt_str:
            df[c] = df[c].astype('object').fillna('NA').astype('category').cat.codes.astype('int16')
            if c not in cat_list:
                cat_list.append(c)
    return df, cat_list

X,      cat_cols_stable = cleanup_object_dtypes(X,      cat_cols_stable)
X_test, _               = cleanup_object_dtypes(X_test, cat_cols_stable)

# Defensive: also handle X_test cat cols whose Polars Null dtype came through
for c in cat_cols_stable:
    if X_test[c].dtype == 'object' or X_test[c].isna().any():
        X_test[c] = X_test[c].fillna('NA').astype('category').cat.codes.astype('int16')

print(f'Feature matrix: {X.shape}, Test: {X_test.shape}')
print(f'Categorical features (final): {len(cat_cols_stable)}')
print(f'X dtype counts: {X.dtypes.value_counts().to_dict()}')

## 5. Gini stability metric

In [ ]:
def gini_stability(week_nums, targets, preds):
    weeks = np.unique(week_nums)
    weekly_gini = []
    for w in weeks:
        mask = week_nums == w
        if targets[mask].sum() == 0:
            continue
        weekly_gini.append(2 * roc_auc_score(targets[mask], preds[mask]) - 1)
    weekly_gini = np.array(weekly_gini)
    mean_gini = weekly_gini.mean()
    slope, intercept, *_ = linregress(np.arange(len(weekly_gini)), weekly_gini)
    residuals = weekly_gini - (intercept + slope * np.arange(len(weekly_gini)))
    return mean_gini + 88.0 * min(0, slope) - 0.5 * residuals.std(), mean_gini, slope, residuals.std()

## 6. Time-based split

In [ ]:
val_mask = week_num > TRAIN_CUTOFF

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
week_val = week_num[val_mask]

print(f'Train: {X_train.shape[0]:,} ({y_train.mean():.3%} pos)')
print(f'Val:   {X_val.shape[0]:,} ({y_val.mean():.3%} pos)')

## 7. Train LightGBM (CPU)

In [ ]:
# LightGBM with best params from Optuna (Sub 5)
lgb_params = {
    'objective': 'binary', 'metric': 'auc',
    'n_estimators': 1000,
    'learning_rate':     0.022812809566861304,
    'num_leaves':        55,
    'min_child_samples': 38,
    'feature_fraction':  0.774816395214887,
    'bagging_fraction':  0.5187196516747448,
    'bagging_freq':      1,
    'reg_alpha':         0.7931156231178,
    'reg_lambda':        0.3782056934716548,
    'scale_pos_weight':  (y_train == 0).sum() / (y_train == 1).sum(),
    'n_jobs': -1, 'verbose': -1, 'random_state': 42,
}

lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)
print(f'LightGBM best iteration: {lgb_model.best_iteration_}')
lgb_val_preds = lgb_model.predict_proba(X_val)[:, 1]
lgb_best_iter = lgb_model.best_iteration_


## 8. Train CatBoost (GPU)

In [ ]:
train_pool = Pool(X_train, label=y_train, cat_features=cat_cols_stable)
val_pool   = Pool(X_val,   label=y_val,   cat_features=cat_cols_stable)

cat_params = {
    'iterations': 1000, 'learning_rate': 0.05, 'depth': 6,
    'l2_leaf_reg': 3.0, 'random_strength': 1.0, 'border_count': 128,
    'auto_class_weights': 'Balanced', 'eval_metric': 'AUC',
    'random_seed': 42, 'task_type': 'GPU', 'devices': '0',
    'verbose': 100,
}

cat_model = CatBoostClassifier(**cat_params)
cat_model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)
print(f'\nCatBoost best iteration: {cat_model.best_iteration_}')

cat_val_preds = cat_model.predict_proba(val_pool)[:, 1]
cat_best_iter = cat_model.best_iteration_

del train_pool, val_pool
gc.collect()

## 9. Compare individual models vs ensemble on validation

In [ ]:
ensemble_val_preds = 0.5 * lgb_val_preds + 0.5 * cat_val_preds

for name, preds in [('LightGBM alone', lgb_val_preds),
                    ('CatBoost alone', cat_val_preds),
                    ('Ensemble 50/50', ensemble_val_preds)]:
    auc = roc_auc_score(y_val, preds)
    score, mean_gini, slope, resid_std = gini_stability(week_val, y_val, preds)
    print(f'{name:18s}  AUC={auc:.4f}  Gini-stab={score:.4f}  mean-gini={mean_gini:.4f}  resid-std={resid_std:.4f}')


## 10. Retrain both models on full data & generate ensembled submission

In [ ]:
# Free validation predictions and per-model objects from training before retraining
del lgb_model, cat_model
del X_train, X_val, y_train, y_val
gc.collect()

# --- LightGBM final (with Optuna best params) ---
lgb_final = lgb.LGBMClassifier(**{**lgb_params, 'n_estimators': lgb_best_iter})
lgb_final.fit(X, y)
lgb_test_preds = lgb_final.predict_proba(X_test)[:, 1]
del lgb_final
gc.collect()
print(f'LightGBM test predictions: shape {lgb_test_preds.shape}, mean {lgb_test_preds.mean():.3f}')

# --- CatBoost final ---
full_pool = Pool(X, label=y, cat_features=cat_cols_stable)
test_pool = Pool(X_test,      cat_features=cat_cols_stable)

cat_final = CatBoostClassifier(**{**cat_params, 'iterations': cat_best_iter})
cat_final.fit(full_pool)
cat_test_preds = cat_final.predict_proba(test_pool)[:, 1]
del cat_final, full_pool, test_pool
gc.collect()
print(f'CatBoost test predictions: shape {cat_test_preds.shape}, mean {cat_test_preds.mean():.3f}')

# --- Blend ---
ensemble_test_preds = 0.5 * lgb_test_preds + 0.5 * cat_test_preds

submission = pl.DataFrame({
    'case_id': test_base['case_id'],
    'score': ensemble_test_preds
})
submission.write_csv(f'{OUTPUT}/submission.csv')
print(f'\nSaved {len(submission):,} ensembled predictions to {OUTPUT}/submission.csv')
print(submission.describe())